In [ ]:
import pyaudio
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft

# to display in separate Tk window
%matplotlib tk

# constants
CHUNK = 1024 * 2            # samples per frame
FORMAT = pyaudio.paInt16     # audio format (bytes per sample?)
CHANNELS = 1                 # single channel for microphone
RATE = 44100                 # samples per second

# create matplotlib figure and axes
fig, (ax1, ax2) = plt.subplots(2, figsize=(15, 7))

# pyaudio class instance
p = pyaudio.PyAudio()

# stream object to get data from microphone
stream = p.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    output=True,
    frames_per_buffer=CHUNK
)

# variable for plotting
x = np.arange(0, 2 * CHUNK, 2)       # samples (waveform)
xf = np.linspace(0, RATE, CHUNK)     # frequencies (spectrum)

# create a line object with random data
line, = ax1.plot(x, np.random.rand(CHUNK), '-', lw=0.5)

# create semilogx line for spectrum
line_fft, = ax2.plot(xf, np.random.rand(CHUNK), '-', lw=0.5)

# Signal range is -32k to 32k for 16bit
AMPLITUDE_LIMIT = 32000

# format waveform axes
ax1.set_title('AUDIO WAVEFORM')
ax1.set_xlabel('Samples')
ax1.set_ylabel('Volume')
ax1.set_ylim(-AMPLITUDE_LIMIT, AMPLITUDE_LIMIT)
ax1.set_xlim(0, 2 * CHUNK)

plt.setp(ax1, xticks=[0, CHUNK, 2 * CHUNK], yticks=[-AMPLITUDE_LIMIT, 0, AMPLITUDE_LIMIT])
plt.grid(linestyle='-', linewidth=1)
ax2.set_xlabel('Frequenz in [Hz]')
ax2.set_ylabel('Amplitude in [dB]')
# format spectrum axes
ax2.set_xlim(0, RATE / 2)
ax2.set_ylim(-100, 40)
print('stream started')

while True:
    # binary data
    data = stream.read(CHUNK, exception_on_overflow=False)   

    data_np = np.frombuffer(data, dtype='h')
    
    line.set_ydata(data_np)
    
    # compute FFT and update line
    yf = fft(data_np)
    line_fft.set_ydata(20*np.log10(np.maximum(np.abs(yf[0:CHUNK]),1e-10)  / (512 * CHUNK)))
    
    fig.canvas.draw()
    fig.canvas.flush_events()